In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DecimalType, TimestampType)

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA retail")

VOLUME_PATH = "/Volumes/workspace/retail/raw_files"
display(dbutils.fs.ls(VOLUME_PATH))

In [0]:
schema = StructType([
    StructField("invoice",      StringType(),       True),
    StructField("stock_code",   StringType(),       True),
    StructField("description",  StringType(),       True),
    StructField("quantity",     IntegerType(),      True),
    StructField("invoice_date", TimestampType(),    True),
    StructField("price",        DecimalType(12, 4), True),
    StructField("customer_id",  StringType(),       True),
    StructField("country",      StringType(),       True),
])

In [0]:
files = {
    "2009_2010": f"{VOLUME_PATH}/online_retail_2009_2010.csv",
    "2010_2011": f"{VOLUME_PATH}/online_retail_2010_2011.csv",
}

def read_csv(path):
    return (
        spark.read\
        .format("csv")\
        .option("header", True)\
        .option("mode", "FAILFAST")   # stop loudly if a row doesn't fit the schema\
        .schema(schema)\
        .load(path)\
        .withColumn("source_file", F.col("_metadata.file_path"))\
        .withColumn("ingested_at", F.current_timestamp())
    )

df_1 = read_csv(files["2009_2010"])
df_2 = read_csv(files["2010_2011"])

# unionByName matches columns by name, which is safer than union()
df_all = df_1.unionByName(df_2)

# Added after the union so ids are unique across both files
df_all = df_all.withColumn("source_row_id", F.monotonically_increasing_id())

df_all.printSchema()

In [0]:
(
    df_all.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.retail.bronze_online_retail")
)

In [0]:
bronze = spark.table("workspace.retail.bronze_online_retail")

display(
    bronze.groupBy("source_file").count()
)

display(
    bronze.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in bronze.columns
    ])
)

display(bronze.limit(10))
